In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

**Creating widget for dinamically inserting files from volume to bronze layer**

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

**Giving path for the source**

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f'/Volumes/source_catalog/landing/child_company_file/{data_source}/*.csv'

print(base_path)

**Creating a dataframe for our source csv**

In [0]:
df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

display(df.limit(10))

**Saving the table as a delta table in bronze schema**

In [0]:
df.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

### Silver Processing

In [0]:
# creating a dataframe from the bronze table
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

In [0]:
# checking for duplicate
df_duplicates = df_bronze.groupBy("customer_id").count().filter(F.col("count") > 1)
display(df_duplicates)

In [0]:
# Deleting the duplicate records

print('Rows before duplicates dropped: ', df_bronze.count())

df_silver = df_bronze.dropDuplicates(['customer_id'])

print('Rows after duplicates dropped: ', df_silver.count())

In [0]:
# check those values which have spaces before the name 
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
# replaces customer_name with its trimmed version

df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)

In [0]:
# Check for the distinct city

df_silver.select("city").distinct().show()

In [0]:
# Inserting correct city names and making only allowed to insert the correct names of the city


city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDelhii': 'New Delhi',
    'NewDelhee': 'New Delhi'
}


allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = (
    df_silver
    .replace(city_mapping, subset=["city"])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)
         .when(F.col("city").isin(allowed), F.col("city"))
         .otherwise(None)
    )
)

In [0]:
# Sanity check
df_silver.select("city").distinct().show()

In [0]:
df_silver.select("customer_name").distinct().show()

In [0]:
# Title case fix
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
     .otherwise(F.initcap("customer_name"))
)

In [0]:
# Check the null city rows


df_silver.filter(F.col("city").isNull()).show(truncate=False)

In [0]:
null_customer_names = [
    'Sprintx Nutrition',
    'Zenathlete Foods',
    'Primefuel Nutrition',
    'Recovery Lane'
]

df_silver.filter(
    F.col('customer_name').isin(null_customer_names)
).show(truncate=False)

In [0]:
# Business Confirmation Note: City corrections confirmed by business team
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",
    789522: "New Delhi",

    # Recovery Lane
    789603: "Hyderabad"
}

In [0]:
# Creating a daraframe from the inserted values


df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

display(df_fix)

In [0]:
# Joining and fixing the null cities

df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn(
        "city",
        F.coalesce("city", "fixed_city")  # Replace null with fixed city
    )
    .drop("fixed_city")
)

In [0]:
display(df_silver)

In [0]:
# Converting the customer ID column to string so it matches our gold layer tables 


df_silver = df_silver.withColumn(
    "customer_id",
    F.col("customer_id").cast("string")
)

print(df_silver.printSchema())

In [0]:
# Hardcoding some columns according to the project reference


df_silver = (
    df_silver

    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )

    # Static attributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

In [0]:
display(df_silver)


In [0]:
# Saving the bronze table to silver


df_silver.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

### Gold processing

In [0]:
df_silver = spark.sql(
    f"SELECT * FROM {catalog}.{silver_schema}.{data_source}"
)

# Take required columns only
df_gold = df_silver.select(
    "customer_id",
    "customer_name",
    "city",
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
# Save it to the gold schema


df_gold.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
# Existing Gold target table
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")

# Prepare the source data from sb_dim_customers
df_child_customers = (
    spark.table("fmcg.gold.sb_dim_customers")
    .select(
        F.col("customer_id").alias("customer_code"),
        "customer",
        "market",
        "platform",
        "channel"
    )
)    

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
